In [ ]:
!pip install numpy pandas matplotlib scipy

## Load all NCU data

In [ ]:
import pandas as pd

metrics_df_raw: pd.DataFrame = pd.read_csv('metrics/metrics_all_ncu.csv')
metrics_df_raw

In [ ]:
# convert stall cols to proportions instead of raw sample counts
stall_cols = [c for c in metrics_df_raw.columns if c.startswith('smsp__pcsamp_warps_issue_stalled_')]
metrics_df = metrics_df_raw.copy()
total_samples = metrics_df_raw[stall_cols].sum(axis=1)
metrics_df[stall_cols] = metrics_df_raw[stall_cols].div(total_samples, axis=0)
metrics_df

In [ ]:
metrics_df.info()

In [ ]:
LABEL_COLS = ['GpuName', 'SeqLen', 'HeadDim', 'Run', 'Action', 'Range']

In [ ]:
RAW_CYCLE_COLS = {
    'SM_Util': 'sm__cycles_active.avg',
    'Tensor_Util': 'sm__pipe_tensor_cycles_active.avg',
    'FMA_Util': 'sm__pipe_fma_cycles_active.avg',
    'FMA_Heavy_Util': 'sm__pipe_fmaheavy_cycles_active.avg',
    'FMA_Lite_Util': 'sm__pipe_fmalite_cycles_active.avg',
    'Shared_Util': 'sm__pipe_shared_cycles_active.avg',
    'ALU_Util': 'sm__pipe_alu_cycles_active.avg',
    'ALU_Heavy_Util': 'sm__pipe_aluheavy_cycles_active.avg',
    'FP64_Util': 'sm__pipe_fp64_cycles_active.avg',
    'L1_Cache_Util': 'l1tex__cycles_active.avg',
    'LTS_Util': 'lts__cycles_active.avg',
    'DRAM_Util': 'dram__cycles_active.avg',
}

# Percent-of-peak metrics require weighted-average aggregation:
#   weighted_util = sum((U_i / 100) * elapsed_i) / sum(elapsed_i)
PCT_COLS = {
    'SFU_Util': 'smsp__inst_executed_pipe_xu.avg.pct_of_peak_sustained_elapsed',
    'SMSP_Util': 'smsp__issue_active.avg.pct_of_peak_sustained_elapsed',
}

In [ ]:
metrics_df_run0 = metrics_df[metrics_df['Run'] == 0].reset_index(drop=True)
metrics_df_run1 = metrics_df[metrics_df['Run'] == 1].reset_index(drop=True)
metrics_df_run2 = metrics_df[metrics_df['Run'] == 2].reset_index(drop=True)
metrics_df_run0

## Compare runs for sanity check

In [ ]:
import numpy as np
from scipy.stats import variation

data_cols = [c for c in metrics_df_run0.columns if c not in LABEL_COLS]

# Stack runs along axis 0 → shape (3, n_rows, n_data_cols)
stacked = np.stack([
    metrics_df_run0[data_cols].values,
    metrics_df_run1[data_cols].values,
    metrics_df_run2[data_cols].values
], axis=0)

# CV = std/mean across the 3 runs for each (row, metric) pair
diff_df = metrics_df_run0[LABEL_COLS].copy()
diff_df[data_cols] = variation(stacked, axis=0)

diff_df

In [ ]:
threshold = 0.03
data_cols = [c for c in diff_df.columns if c not in LABEL_COLS]

flagged = (
    diff_df[data_cols]
    .stack()
    .reset_index()
    .rename(columns={'level_1': 'metric', 0: 'cv'})
    .query('cv >= @threshold')
)

label_vals = diff_df.loc[flagged['level_0'], LABEL_COLS].reset_index(drop=True)
flagged_df = pd.concat([label_vals, flagged[['metric', 'cv']].reset_index(drop=True)], axis=1)
flagged_df

## Get Utilization Percentages

### Individual Kernels

In [ ]:
sm_cycles_elapsed = metrics_df_run0['sm__cycles_elapsed.avg']

util_df = metrics_df_run0[LABEL_COLS].copy()

for util_col, raw_col in RAW_CYCLE_COLS.items():
    util_df[util_col] = metrics_df_run0[raw_col] / sm_cycles_elapsed

for util_col, raw_col in PCT_COLS.items():
    util_df[util_col] = metrics_df_run0[raw_col] / 100.0

util_df

In [ ]:
util_df.to_csv('metrics/util_ncu.csv')

### Aggregate across multiple kernels in same backend

In [ ]:
GROUP_COLS = ['GpuName', 'SeqLen', 'HeadDim']

In [ ]:
def aggregate_util(df, group_cols, util_cols, pct_cols):
    """
    Aggregate cycle-based and pct-of-peak utilization metrics across kernels.

    util_cols: dict of {util_col_name: raw_cycle_col}
    pct_cols:  dict of {util_col_name: raw_pct_of_peak_col}

    Uses min_count=1 in groupby sum so all-NaN groups stay NaN.
    """
    df = df.copy()
    for util_name, raw_col in pct_cols.items():
        df[f'_weighted_{util_name}'] = df[raw_col] * df['sm__cycles_elapsed.avg']

    weighted_cols = [f'_weighted_{k}' for k in pct_cols]
    agg_cols = list(util_cols.values()) + ['sm__cycles_elapsed.avg'] + weighted_cols

    agg = (
        df[group_cols + agg_cols]
        .groupby(group_cols)
        .sum(min_count=1)
        .reset_index()
    )

    result = agg[group_cols].copy()
    for util_col, raw_col in util_cols.items():
        result[util_col] = agg[raw_col] / agg['sm__cycles_elapsed.avg']

    for util_name in pct_cols:
        result[util_name] = (
                agg[f'_weighted_{util_name}'] / agg['sm__cycles_elapsed.avg'] / 100
        )

    return result

#### Math backend

In [ ]:
exclude_prefixes = ('fmha_cutlassF_', 'flash_fwd_', 'cudnn_')
metrics_df_run0_math_only = metrics_df_run0[~metrics_df_run0['Action'].str.startswith(exclude_prefixes)].reset_index(
    drop=True)
metrics_df_run0_math_only.head(31)

In [ ]:
util_df_math_only = aggregate_util(
    metrics_df_run0_math_only,
    group_cols=GROUP_COLS,
    util_cols=RAW_CYCLE_COLS,
    pct_cols=PCT_COLS,
)
util_df_math_only

In [ ]:
util_df_math_only.to_csv('metrics/util_ncu_math.csv')

#### FlashAttention backend

In [ ]:
metrics_df_run0_flash_only = metrics_df_run0[metrics_df_run0['Action'].str.startswith('flash_fwd_')].reset_index(
    drop=True)
metrics_df_run0_flash_only.head()

In [ ]:
util_df_flash_only = aggregate_util(
    metrics_df_run0_flash_only,
    group_cols=GROUP_COLS,
    util_cols=RAW_CYCLE_COLS,
    pct_cols=PCT_COLS,
)
util_df_flash_only

In [ ]:
util_df_flash_only.to_csv('metrics/util_ncu_flash.csv')

## Plot Setup

In [ ]:
import os
import matplotlib.pyplot as plt

In [ ]:
UNIT_COLS = sorted([col for col in util_df.columns if col.endswith('_Util')])
UNIT_COLS

In [ ]:
UNIT_LABELS = [col[:col.index('_Util')] for col in UNIT_COLS]
UNIT_LABELS

In [ ]:
GPU_LABELS = {
    'rtx2060': 'RTX 2060',
    't4': 'T4',
    'a100-sxm4-40gb': 'A100',
    'l4': 'L4',
    'h10080gbhbm3': 'H100',
    'rtxpro6000blackwellserveredition': 'Blackwell',
}
GPU_ORDER = ['RTX 2060', 'T4', 'A100', 'L4', 'H100', 'Blackwell']

In [ ]:
SEQ_LENS = metrics_df_run0['SeqLen'].unique()
HEAD_DIMS = metrics_df_run0['HeadDim'].unique()

## Plot Relative Durations of Kernels

#### Math backend

In [ ]:
def _plot_kernel_durations(df, gpu_name, seq_len, head_dim,
                           title_prefix, save_dir,
                           show=False, save=False):
    reverse_gpu_labels = {v: k for k, v in GPU_LABELS.items()}
    raw_gpu_name = reverse_gpu_labels[gpu_name]

    sub = df[
        (df['GpuName'] == raw_gpu_name) &
        (df['SeqLen'] == seq_len) &
        (df['HeadDim'] == head_dim)
        ].copy()

    assert not sub.empty, (
        f"No {title_prefix} kernel data for gpu_name='{gpu_name}', "
        f"seq_len={seq_len}, head_dim={head_dim}"
    )

    agg = (sub.groupby('Action')['sm__cycles_elapsed.avg']
           .sum()
           .sort_values(ascending=False))

    fig, ax = plt.subplots(figsize=(12, 5))
    ax.bar(range(len(agg)), agg.values)
    ax.set_xticks(range(len(agg)))
    ax.set_xticklabels(agg.index, rotation=45, ha='right', fontsize=8)
    ax.set_title(f'{title_prefix} Kernel Durations \u2014 {gpu_name}, SeqLen={seq_len}, HeadDim={head_dim}')
    ax.set_xlabel('Kernel')
    ax.set_ylabel('Total sm__cycles_elapsed.avg (cycles)')
    fig.tight_layout()

    if save:
        os.makedirs(f'plots/{save_dir}', exist_ok=True)
        plt.savefig(f'plots/{save_dir}/{gpu_name}_{seq_len}x{head_dim}.png')

    if show:
        plt.show()
    else:
        plt.close()

In [ ]:
def plot_math_kernel_durations(gpu_name: str, seq_len: int, head_dim: int,
                               show: bool = False, save: bool = False):
    """
    Bar chart of total sm__cycles_elapsed.avg per Math-backend kernel name
    for a single (gpu_name, seq_len, head_dim) config.

    Excludes fmha_cutlassF_ (xformers/MEA), flash_fwd_ (FA2), and cudnn_ kernels.
    Multiple invocations of the same kernel name are summed.
    """
    _plot_kernel_durations(
        metrics_df_run0_math_only, gpu_name, seq_len, head_dim,
        title_prefix='Math', save_dir='math_kernel_durations',
        show=show, save=save,
    )

In [ ]:
# Example
plot_math_kernel_durations('A100', 2048, 128, show=True)

In [ ]:
for gpu_name in GPU_ORDER:
    for seq_len in SEQ_LENS:
        for head_dim in HEAD_DIMS:
            try:
                plot_math_kernel_durations(gpu_name, seq_len, head_dim, save=True)
            except Exception as e:
                print(e)

Note that `volta_sgemm_*`, `ampere_sgemm_*`, `sm80_xmma_gemm_`, and `Kernel2` kernels are all Matrix Mult

#### FlashAttention-2 Backend

In [ ]:
def plot_flash_kernel_durations(gpu_name: str, seq_len: int, head_dim: int,
                                show: bool = False, save: bool = False):
    """
    Bar chart of total sm__cycles_elapsed.avg per FA2 kernel name
    for a single (gpu_name, seq_len, head_dim) config.

    Includes flash_fwd_splitkv_combine_kernel (the reduction step) since
    this is a duration plot and it contributes to total runtime.
    Multiple invocations of the same kernel name are summed.
    """
    _plot_kernel_durations(
        metrics_df_run0_flash_only, gpu_name, seq_len, head_dim,
        title_prefix='FA2', save_dir='flash_kernel_durations',
        show=show, save=save,
    )

In [ ]:
# Example
plot_flash_kernel_durations('A100', 2048, 128, show=True)

In [ ]:
for gpu_name in GPU_ORDER:
    for seq_len in SEQ_LENS:
        for head_dim in HEAD_DIMS:
            try:
                plot_flash_kernel_durations(gpu_name, seq_len, head_dim, save=True)
            except Exception as e:
                print(e)

## Plot Utilization of Function Units

### Setup

In [ ]:
KERNEL_ORDER = ['Math', 'xformers', 'FA2', 'cuDNN']


def label_kernel(name: str):
    """
    Only needed for CuDNN and MEA.
    Math and FA2 backends are covered separately because they are aggregated.
    """
    if 'cudnn_generated' in name:
        return 'cuDNN'
    if 'fmha_cutlassF' in name:
        return 'xformers'
    return None

In [ ]:
# xformers and cuDNN: per-kernel rows from util_df, averaged over invocations (should only be 1 each)
plot_df = util_df.copy()
plot_df['KernelLabel'] = plot_df['Action'].apply(label_kernel)
plot_df['GpuLabel'] = plot_df['GpuName'].map(GPU_LABELS)
plot_df = plot_df[
    plot_df['KernelLabel'].isin(['xformers', 'cuDNN']) &
    plot_df['GpuLabel'].isin(GPU_ORDER)
    ].copy()
xformers_cudnn_agg = (plot_df
                      .groupby(['GpuLabel', 'KernelLabel', 'SeqLen', 'HeadDim'])[UNIT_COLS]
                      .mean()
                      .reset_index())

# FA2: cycle-weighted aggregate from util_df_flash_only
fa2_df = util_df_flash_only.copy()
fa2_df['KernelLabel'] = 'FA2'
fa2_df['GpuLabel'] = fa2_df['GpuName'].map(GPU_LABELS)
fa2_df = fa2_df[fa2_df['GpuLabel'].isin(GPU_ORDER)][
    ['GpuLabel', 'KernelLabel', 'SeqLen', 'HeadDim'] + UNIT_COLS
    ]

# Math: cycle-weighted aggregate from util_df_math_only
math_df = util_df_math_only.copy()
math_df['KernelLabel'] = 'Math'
math_df['GpuLabel'] = math_df['GpuName'].map(GPU_LABELS)
math_df = math_df[math_df['GpuLabel'].isin(GPU_ORDER)][
    ['GpuLabel', 'KernelLabel', 'SeqLen', 'HeadDim'] + UNIT_COLS
    ]

agg_df = pd.concat([xformers_cudnn_agg, fa2_df, math_df], ignore_index=True)

print(f"agg_df shape: {agg_df.shape}")
print(agg_df.groupby(['KernelLabel', 'GpuLabel']).size().unstack(fill_value=0))

### Plot util of each Function Unit across GPU and Kernel for certain Seq Len, Head Dim

In [ ]:
def plot_util_by_gpu_and_kernel(seq_len: int, head_dim: int, util_col: str, show: bool = False,
                                save: bool = False):
    """
    Bar chart of util_col across kernels and GPUs for a single (seq_len, head_dim) config.
    """
    sub = agg_df[(agg_df['SeqLen'] == seq_len) & (agg_df['HeadDim'] == head_dim)]
    pivot = (sub
             .pivot_table(index='KernelLabel', columns='GpuLabel', values=util_col)
             .reindex(index=KERNEL_ORDER, columns=GPU_ORDER))

    col_label = dict(zip(UNIT_COLS, UNIT_LABELS)).get(util_col, util_col)

    fig, ax = plt.subplots(figsize=(7, 4))
    pivot.plot(kind='bar', ax=ax, width=0.75, rot=25)
    ax.set_title(f'{col_label} Utilization \u2014 SeqLen={seq_len}, HeadDim={head_dim}')
    ax.set_xlabel('')
    ax.set_ylabel(f'{col_label} Utilization (fraction of cycles)')
    ax.set_ylim(0, 1)
    ax.legend(title='GPU', fontsize=8)
    fig.tight_layout()

    if not os.path.exists(f'plots'):
        os.makedirs(f'plots')
    if not os.path.exists(f'plots/util_across_gpu_and_kernel'):
        os.makedirs(f'plots/util_across_gpu_and_kernel')
    if not os.path.exists(f'plots/util_across_gpu_and_kernel/{util_col}'):
        os.makedirs(f'plots/util_across_gpu_and_kernel/{util_col}')

    if save:
        plt.savefig(f'plots/util_across_gpu_and_kernel/{util_col}/{util_col}_{seq_len}x{head_dim}.png')

    if show:
        plt.show()
    else:
        plt.close()

In [ ]:
# Example
plot_util_by_gpu_and_kernel(2048, 128, 'Tensor_Util', show=True)

In [ ]:
for unit_col in UNIT_COLS:
    print(f"Plotting {unit_col}")
    for seq_len in SEQ_LENS:
        print(f"  Seq len {seq_len}")
        for head_dim in HEAD_DIMS:
            print(f"    Head dim {head_dim}")
            plot_util_by_gpu_and_kernel(seq_len, head_dim, unit_col, save=True)

### Plot util of all Function Units for certain GPU, Kernel, Seq Len, Head Dim

In [ ]:
def plot_util_all_func_units(gpu_name: str, kernel_name: str, seq_len: int, head_dim: int,
                             show: bool = False, save: bool = False):
    """
    Bar chart of all util_cols for a single (GPU, kernel, seq_len, head_dim) config.
    """
    sub = agg_df[
        (agg_df['GpuLabel'] == gpu_name) &
        (agg_df['KernelLabel'] == kernel_name) &
        (agg_df['SeqLen'] == seq_len) &
        (agg_df['HeadDim'] == head_dim)
        ]

    assert not sub.empty, (
        f"No data for gpu_name='{gpu_name}', kernel_name='{kernel_name}', "
        f"seq_len={seq_len}, head_dim={head_dim}"
    )

    values = sub[UNIT_COLS].iloc[0]

    fig, ax = plt.subplots(figsize=(10, 4))
    ax.bar(UNIT_LABELS, values)
    ax.set_title(f'Functional Unit Utilization — {gpu_name}, {kernel_name}, SeqLen={seq_len}, HeadDim={head_dim}')
    ax.set_xlabel('Functional Unit')
    ax.set_ylabel('Utilization (fraction of cycles)')
    ax.set_ylim(0, 1)
    ax.tick_params(axis='x', rotation=30)
    fig.tight_layout()

    if not os.path.exists(f'plots'):
        os.makedirs(f'plots')
    if not os.path.exists(f'plots/util_all_func_units'):
        os.makedirs(f'plots/util_all_func_units')
    if not os.path.exists(f'plots/util_all_func_units/{gpu_name}'):
        os.makedirs(f'plots/util_all_func_units/{gpu_name}')

    if save:
        plt.savefig(f'plots/util_all_func_units/{gpu_name}/{gpu_name}_{kernel_name}_{seq_len}x{head_dim}.png')

    if show:
        plt.show()
    else:
        plt.close()

In [ ]:
# Example
plot_util_all_func_units('A100', 'Math', 2048, 128, show=True)

In [ ]:
for gpu_name in GPU_ORDER:
    print(f"Plotting {gpu_name}")
    for kernel_name in KERNEL_ORDER:
        print(f"  Kernel {kernel_name}")
        for seq_len in SEQ_LENS:
            print(f"    Seq len {seq_len}")
            for head_dim in HEAD_DIMS:
                print(f"      Head dim {head_dim}")
                try:
                    plot_util_all_func_units(gpu_name, kernel_name, seq_len, head_dim, save=True)
                except AssertionError as e:
                    print(f'      {e}')